In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))
from component2_wind_spraying.drift_model.features import FEATURES, DROPLET_CODE, VALID_RANGES

DATA = Path.cwd() / "data"
model = XGBRegressor()
model.load_model(str(ROOT / "component2_wind_spraying" / "drift_model" / "saved" / "drift_xgb.json"))
print("model loaded | features:", FEATURES)

model loaded | features: ['crosswind_ms', 'height_m', 'swath_m', 'temp_c', 'rh_pct', 'droplet_code']


In [2]:
raw = pd.concat([pd.read_excel(f) for f in sorted(DATA.glob("*FlightRecord.xlsx"))], ignore_index=True)
print(raw.shape)
raw[["Flight time", "Location", "Task Type", "Height", "Speed", "Row Spacing", "Sprayed area"]].head()

(981, 21)


,Flight time,Location,Task Type,Height,Speed,Row Spacing,Sprayed area
0,2026-08-20 23:58:37-00:03:24,"Sungavila, Polonnaruwa District, North Central...",Spraying,NaN,NaN,NaN,3426.666684
1,2026-08-20 23:56:59-23:58:37,"Sungavila, Polonnaruwa District, North Central...",Spraying,NaN,NaN,NaN,400.000002
2,2026-08-20 23:52:15-23:53:14,"Sungavila, Polonnaruwa District, North Central...",Spraying,NaN,NaN,NaN,253.333335
3,2026-08-20 23:46:50-23:52:14,"Sungavila, Polonnaruwa District, North Central...",Spraying,NaN,NaN,NaN,3140.000016
4,2026-08-20 23:40:11-23:44:19,"Sungavila, Polonnaruwa District, North Central...",Spraying,NaN,NaN,NaN,3306.666683


In [3]:
steps = {"all records": len(raw)}
f = raw.drop_duplicates(subset=["Flight time", "Serial Number"])
steps["remove duplicates"] = len(f)
f = f[f["Task Type"] == "Spraying"]
steps["spraying only"] = len(f)
f = f[f["Sprayed area"] > 0]
steps["sprayed area > 0"] = len(f)
f = f.dropna(subset=["Height", "Speed"])
f = f[f["Row Spacing"] > 0]
steps["has height, speed, swath"] = len(f)
f = f.assign(start=pd.to_datetime(f["Flight time"].astype(str).str[:19]), district=f["Location"].str.extract(r"([A-Za-z ]+) District")[0].str.strip())
f = f[f["start"].dt.hour.between(6, 18)]
steps["daytime 06-18 h"] = len(f)
print(f["district"].value_counts(dropna=False))
pd.Series(steps, name="flights")

district
Ampara         374
Badulla        123
Polonnaruwa     67
Name: count, dtype: int64


all records                 981
remove duplicates           914
spraying only               898
sprayed area > 0            849
has height, speed, swath    593
daytime 06-18 h             564
Name: flights, dtype: int64